In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as si
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize

from utilsVanilla import *

In [ ]:
S0 = 100
V0 = 0.04
r = 0.05
rho = -0.5
kappa = 1.5
theta = 0.04
sigma = 0.25 
xi = 0.25   
M = 15000
dt = 0.001
seed = 42

K_list = np.linspace(75, 125, 5)  # Strike prices
T_list = np.linspace(0.2, 2, 5)  # Maturity times

In [ ]:
pricefield = grid(S0, V0, r, kappa, theta, sigma, rho, K_list, T_list, M, dt, seed)

price_mc, price_analytic, iv_mc, iv_analytic = build_surfaces(pricefield ,S0, r, K_list, T_list)

# Plot call price surfaces
visualize_surface(K_list, T_list, price_mc, "MC Heston Call Price Surface", "Call Price")
visualize_surface(K_list, T_list, price_analytic, "Analytic Heston Call Price Surface", "Call Price")

# Plot implied volatility surfaces
visualize_surface(K_list, T_list, iv_mc, "MC Implied Volatility Surface", "Implied Volatility")
visualize_surface(K_list, T_list, iv_analytic, "Analytic Implied Volatility Surface", "Implied Volatility")

In [ ]:
S0 = 4237.86
r = 0.05  # as per assignment

raw = np.load("data/raw_ivol_surfaces.npy", allow_pickle=True).item()
dates = sorted(list(raw.keys()))
print("Available dates:", dates)

# Pick two dates for diagnostics (for example, first and last available)
date1 = dates[0]
date2 = dates[-1]

mkt1 = raw[date1]
mkt2 = raw[date2]

Tlist1 = mkt1['tenors']        # (N,)
Klist1 = mkt1['strikes'][:, 0] # (15,); strikes for all maturities are the same in raw data
ivol1 = mkt1['vols']           # (15, N)

Tlist2 = mkt2['tenors']
Klist2 = mkt2['strikes'][:, 0]
ivol2 = mkt2['vols']

In [ ]:
theta0 = [-0.5, 0.04, 0.04, 1.5, 0.3]
bounds = [(-0.999, 0.999),   # ro
          (1e-5, 0.5),       # V0
          (1e-5, 0.5),       # t (theta)
          (1e-3, 10.0),      # k (kappa)
          (1e-3, 2.0)]       # s (sigma)

print("Calibrating for date1:", date1)
res1 = minimize(
    calib_obj, theta0, args=(Klist1, Tlist1, ivol1, S0, r), bounds=bounds, method='L-BFGS-B',
    options={'maxiter': 80, 'disp': True}
)
print("Optimal params date1:", res1.x)

print("\nCalibrating for date2:", date2)
res2 = minimize(
    calib_obj, theta0, args=(Klist2, Tlist2, ivol2, S0, r), bounds=bounds, method='L-BFGS-B',
    options={'maxiter': 80, 'disp': True}
)
print("Optimal params date2:", res2.x)

In [ ]:
k_axis   = log_moneyness_axis(Klist1, S0)   # log-moneyness
m_axis   = rel_moneyness_axis(Klist1, S0)   # K/S0 − 1

# plots for market IV surface for date1

# (a) log-moneyness 
plot_iv_heatmap_custom(k_axis, Tlist1, ivol1,
                       xlab='log-moneyness  $k=\\ln(K/S_0)$',
                       title=f'Market IV {date1}  (log-moneyness)',
                       xlim=(-0.75, 0.75))

# (b) relative moneyness 
plot_iv_heatmap_custom(m_axis, Tlist1, ivol1,
                       xlab='relative moneyness  $\\tilde m = K/S_0-1$',
                       title=f'Market IV {date1}  (relative moneyness)',
                       xlim=(-0.5, 1.0))

In [ ]:
iv_model1 = heston_surface_iv(res1.x, S0, r, Klist1, Tlist1)
iv_model2 = heston_surface_iv(res2.x, S0, r, Klist2, Tlist2)

rmse1, mae1, maxerr1 = error_metrics(ivol1, iv_model1)
rmse2, mae2, maxerr2 = error_metrics(ivol2, iv_model2)

print(f"{date1}: RMSE={rmse1:.4f}, MAE={mae1:.4f}, MaxErr={maxerr1:.4f}")
print(f"{date2}: RMSE={rmse2:.4f}, MAE={mae2:.4f}, MaxErr={maxerr2:.4f}")